# sweep-v1 -- settings and one new rule on top of eat-rest-v1 (baseline untouched)

Mechanism study, not a scored experiment: nothing is written to `results/`. The preserved baseline
file is never edited; variants are built by passing keyword overrides to its policy class
(`external/candidates/sweep_config.py`), and the new rule lives in its own candidate folder
(`external/candidates/eat-rest-fedbirth`, which loads the baseline unmodified and subclasses it).

Why these knobs: a death diagnosis of v1 on validation seeds 2001/2005/2006/2010 showed (a) the population
is capped at ~6 young agents shrinking to a floor of 2, so the game ends within seconds once 3-4 agents have
a bad streak, (b) late-game newborns (75 energy, ~20 s of fuel) starve at age 13-27 while adults hold 200+
energy, (c) predators kill 30-77 agents per game throughout.

Progress lines appear as games finish: `name: mean s (n games, paired delta vs the first variant)`.
Noise warning: paired differences on 16 seeds have a standard error of ~60-80 s. Treat anything under
~+150 s as unproven and confirm on the second seed block before believing it.

**Cluster setup:** same as `test-eat-rest-v1.ipynb` (`.env` with `GITHUB_TOKEN=<token>`).
If `rl-v1` training is still running it is using all 40 CPUs; interrupt it first or these games crawl.

In [1]:
import os

CLONE_DIR = "/home/jovyan/Nordic-AI-cup-2026"
if os.path.isdir(os.path.join(CLONE_DIR, ".git")):
    print(f"{CLONE_DIR} already cloned - skipping (use `git pull` there to update)")
else:
    # GitHub token is read from a git-ignored .env (GITHUB_TOKEN=...) in the kernel's cwd, or from the environment
    if os.path.isfile(".env"):
        for line in open(".env"):
            key, sep, value = line.strip().partition("=")
            if sep and not key.startswith("#"):
                os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))
    TOKEN = os.environ.get("GITHUB_TOKEN")
    if not TOKEN:
        raise RuntimeError(f"GITHUB_TOKEN not set - create {os.path.abspath('.env')} containing GITHUB_TOKEN=<token>")
    !git clone https://{TOKEN}@github.com/sjoeen/Nordic-AI-cup-2026.git {CLONE_DIR}

/home/jovyan/Nordic-AI-cup-2026 already cloned - skipping (use `git pull` there to update)


In [2]:
import glob
import os
import subprocess
import sys

os.chdir(CLONE_DIR)
!git fetch origin challenge-1V2
!git checkout challenge-1V2
!git pull origin challenge-1V2

# Must run from survival-simulator/ so `src`, `agents`, `training` import.
if os.path.basename(os.getcwd()) != "survival-simulator":
    candidates = sorted({os.path.realpath(p) for p in glob.glob(os.path.join(os.getcwd(), "**", "survival-simulator"), recursive=True)
                         if os.path.isfile(os.path.join(p, "requirements.txt"))})
    if len(candidates) != 1:
        raise RuntimeError(f"cwd is {os.getcwd()}; found {len(candidates)} survival-simulator checkouts {candidates} - %cd into the right one")
    os.chdir(candidates[0])

print("cwd:", os.getcwd())
sys.path.insert(0, os.getcwd())
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 10 (delta 4), reused 10 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (10/10), 7.39 KiB | 1.48 MiB/s, done.
From https://github.com/sjoeen/Nordic-AI-cup-2026
 * branch            challenge-1V2 -> FETCH_HEAD
   23610c7..2ad60ab  challenge-1V2 -> origin/challenge-1V2
M	Nordic-AI-Cup-2026-main/survival-simulator/results/index.csv
Already on 'challenge-1V2'
Your branch is behind 'origin/challenge-1V2' by 1 commit, and can be fast-forwarded.
  (use "git pull" to update your local branch)
From https://github.com/sjoeen/Nordic-AI-cup-2026
 * branch            challenge-1V2 -> FETCH_HEAD
Updating 23610c7..2ad60ab
Fast-forward
 .../candidates/eat-rest-fedbirth/survival_agent.py |  59 ++++++
 .../external/candidates/sweep_config.py            | 105 ++++++++++
 .../survival-simulator/sweep-v1.ipynb              | 223 +++++++++++++++++++++
 3

In [3]:
!{sys.executable} -m pip install -r requirements.txt -r requirements-dev.txt

  Using cached pytest-9.1.1-py3-none-any.whl.metadata (7.6 kB)
  Using cached imageio_ffmpeg-0.6.0-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached iniconfig-2.3.0-py3-none-any.whl.metadata (2.5 kB)
Using cached pytest-9.1.1-py3-none-any.whl (386 kB)
Using cached imageio_ffmpeg-0.6.0-py3-none-manylinux2014_x86_64.whl (29.5 MB)
Using cached iniconfig-2.3.0-py3-none-any.whl (7.5 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pytest]2m2/3 [pytest]-ffmpeg]


In [4]:
import json, subprocess, sys, time
from collections import defaultdict
import pandas as pd

def sweep(candidate, variants, seeds, out, workers=None, every=8):
    """Run external/candidates/sweep_config.py and print a running table as games finish.
    The first variant is the reference; 'delta' is the paired mean difference on seeds both have finished."""
    cmd = [sys.executable, "-u", "external/candidates/sweep_config.py", "--candidate", candidate,
           "--seeds", *seeds, "--out", out, *[x for v in variants for x in ("--variant", v)]]
    if workers: cmd += ["--workers", str(workers)]
    print("running:", " ".join(cmd)); t0 = time.perf_counter()
    names = [v.split(":")[0] for v in variants]; done = defaultdict(dict); n = 0
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        if not line.startswith("{"):
            if "pkg_resources" not in line and "pygame" not in line: print(line, end="")
            continue
        row = json.loads(line); done[row["variant"]][row["seed"]] = row["extinction_time"]; n += 1
        if n % every == 0:
            ref = done[names[0]]
            parts = []
            for name in names:
                t = done[name]
                if not t: continue
                both = [t[s] - ref[s] for s in t if s in ref]
                parts.append(f"{name}: {sum(t.values())/len(t):.0f} s (n={len(t)}" + (f", delta {sum(both)/len(both):+.0f}" if both and name != names[0] else "") + ")")
            print(f"[{(time.perf_counter()-t0)/60:4.1f} min, {n} games]  " + " | ".join(parts))
    proc.wait()
    if proc.returncode: raise RuntimeError(f"sweep failed with exit code {proc.returncode}")
    return pd.read_csv(out)

def table(df):
    wide = df.pivot(index="seed", columns="variant", values="extinction_time")
    ref = wide.columns[0] if "base" not in wide.columns else "base"
    summary = pd.DataFrame({"mean": wide.mean(), "median": wide.median(), "min": wide.min(), "max": wide.max(),
                            "delta_vs_" + ref: wide.sub(wide[ref], axis=0).mean(),
                            "se_of_delta": wide.sub(wide[ref], axis=0).sem(),
                            "seeds_better": wide.gt(wide[ref], axis=0).sum()}).round(0)
    return summary.sort_values("mean", ascending=False), wide

## 1. Population settings of v1 (existing knobs only)

In [5]:
SEEDS = ["2000:2016"]
df_pop = sweep("original-eat-rest-preserved", [
    "base",
    "floor4:population_floor=4",
    "floor6:population_floor=6",
    "decay1800:population_decay=1800",
    "pop9slow:population=9,population_decay=1800,population_floor=4",
], SEEDS, "logs/sweep_population.csv")
summary, wide = table(df_pop); summary

running: /opt/conda/bin/python -u external/candidates/sweep_config.py --candidate original-eat-rest-preserved --seeds 2000:2016 --out logs/sweep_population.csv --variant base --variant floor4:population_floor=4 --variant floor6:population_floor=6 --variant decay1800:population_decay=1800 --variant pop9slow:population=9,population_decay=1800,population_floor=4
[ 3.5 min, 8 games]  base: 1231 s (n=4) | floor4: 970 s (n=3, delta -693) | floor6: 862 s (n=1, delta -245)
[ 4.4 min, 16 games]  base: 1264 s (n=7) | floor4: 1244 s (n=8, delta -10) | floor6: 862 s (n=1, delta -245)
[ 4.9 min, 24 games]  base: 1332 s (n=11) | floor4: 1278 s (n=11, delta -14) | floor6: 996 s (n=2, delta -253)
[ 5.6 min, 32 games]  base: 1360 s (n=14) | floor4: 1296 s (n=15, delta -76) | floor6: 999 s (n=3, delta -318)
[ 7.0 min, 40 games]  base: 1375 s (n=16) | floor4: 1296 s (n=15, delta -81) | floor6: 1100 s (n=8, delta -203) | decay1800: 748 s (n=1, delta -545)
[ 7.8 min, 48 games]  base: 1375 s (n=16) | floor4

,mean,median,min,max,delta_vs_base,se_of_delta,seeds_better
variant,,,,,,,
base,1375.0,1382.0,1107.0,1644.0,0.0,0.0,0
floor4,1310.0,1370.0,600.0,1584.0,-65.0,71.0,7
decay1800,1297.0,1297.0,748.0,1744.0,-77.0,62.0,6
floor6,1139.0,1183.0,862.0,1461.0,-236.0,44.0,1
pop9slow,1112.0,1125.0,691.0,1303.0,-263.0,56.0,1


## 2. New rule: only give birth next to food (`eat-rest-fedbirth`)

`off` is the rule disabled and is byte-for-byte the baseline's behaviour, so it doubles as the reference.

In [6]:
df_fed = sweep("eat-rest-fedbirth", [
    "off:birth_food_distance=0",
    "d100:birth_food_distance=100",
    "d150:birth_food_distance=150",
    "d250:birth_food_distance=250",
    "d150late:birth_food_distance=150,birth_food_start=600",
    "d150pat25:birth_food_distance=150,birth_food_patience=25",
], SEEDS, "logs/sweep_fedbirth.csv")
summary, wide = table(df_fed); summary

running: /opt/conda/bin/python -u external/candidates/sweep_config.py --candidate eat-rest-fedbirth --seeds 2000:2016 --out logs/sweep_fedbirth.csv --variant off:birth_food_distance=0 --variant d100:birth_food_distance=100 --variant d150:birth_food_distance=150 --variant d250:birth_food_distance=250 --variant d150late:birth_food_distance=150,birth_food_start=600 --variant d150pat25:birth_food_distance=150,birth_food_patience=25
[ 4.3 min, 8 games]  off: 1243 s (n=4) | d100: 1024 s (n=4, delta +212)
[ 5.0 min, 16 games]  off: 1297 s (n=7) | d100: 1172 s (n=8, delta -15) | d150: 1762 s (n=1)
[ 5.6 min, 24 games]  off: 1319 s (n=12) | d100: 1182 s (n=10, delta -164) | d150: 1619 s (n=2, delta +233)
[ 6.2 min, 32 games]  off: 1346 s (n=14) | d100: 1188 s (n=12, delta -148) | d150: 1323 s (n=6, delta -112)
[ 8.3 min, 40 games]  off: 1346 s (n=15) | d100: 1278 s (n=15, delta -56) | d150: 1267 s (n=8, delta -96) | d250: 823 s (n=2, delta -427)
[ 9.5 min, 48 games]  off: 1363 s (n=16) | d100: 

,mean,median,min,max,delta_vs_d100,se_of_delta,seeds_better
variant,,,,,,,
d150late,1415.0,1422.0,1115.0,1842.0,108.0,73.0,12
off,1363.0,1354.0,1107.0,1644.0,55.0,81.0,9
d150,1358.0,1452.0,674.0,1762.0,50.0,93.0,9
d150pat25,1327.0,1372.0,699.0,1798.0,20.0,87.0,9
d100,1308.0,1315.0,860.0,1807.0,0.0,0.0,0
d250,1298.0,1306.0,669.0,1681.0,-10.0,80.0,9


## 3. Confirmation on fresh seeds

Put the one or two best variants here (keep the reference first). These seeds were not used to pick them.

In [ ]:
CONFIRM_SEEDS = ["2100:2105"]   # 48 fresh seeds
df_confirm = sweep("eat-rest-fedbirth", [
    "off:birth_food_distance=0",
    "d150:birth_food_distance=150",      # <- replace with the winner(s) from above
], CONFIRM_SEEDS, "logs/sweep_confirm.csv")
summary, wide = table(df_confirm); summary

running: /opt/conda/bin/python -u external/candidates/sweep_config.py --candidate eat-rest-fedbirth --seeds 2100:2148 --out logs/sweep_confirm.csv --variant off:birth_food_distance=0 --variant d150:birth_food_distance=150


## 4. Gene-aware breeding (`eat-rest-selective`)

Local finding before running this: v1 ALREADY selects for speed. On seeds 2003 and 2010 the colony's mean walking
speed rose 10 -> ~13 (500 s) -> ~15-16 (750 s) with the untouched baseline, because it ranks eligible parents by speed
and predators remove slow agents. With the default rule the new candidate vetoed **zero** births in 800 s (every parent
the baseline picks is already within 10% of the best), so `default` should equal `off`. The stricter variants are the
real test. Watch the "mean walking-speed gene" table printed at the end: if it does not rise faster than `off`,
the rule is not selecting anything, whatever the survival numbers say.

In [ ]:
df_sel = sweep("eat-rest-selective", [
    "off:sel_enabled=0",
    "default",
    "strict97:sel_strict_fraction=0.97",
    "best_only:sel_strict_fraction=0.999",
    "strict97long:sel_strict_fraction=0.97,sel_strict_until=1500,sel_strict_min_agents=4",
    "no_sterile:sel_sterile_fraction=0",
], SEEDS, "logs/sweep_selective.csv")
summary, wide = table(df_sel); summary

In [ ]:
# How much the rule actually did, per variant (mean per game)
cols = [c for c in df_sel.columns if c.startswith(("sel_", "speed_t"))]
df_sel.groupby("variant")[cols].mean().round(2)